# Data and model exploration
- Data source 1 [link](https://www.kaggle.com/datasets/andrewmvd/spotify-playlists)
- Data source 2 [link](https://www.kaggle.com/datasets/devdope/900k-spotify/data)

## Big questions
- How can we validate our model is working?
  - Try to predict if recommendations show on other user's playlist, and that rate
  - Have classmates participate, and have them take 10 or so recommendations and tell us what % they like

In [2]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import warnings
import re

# set display options
warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)

In [3]:
# read in data
df = pd.read_csv('../../data/spotify_dataset.csv', on_bad_lines='skip')
df.columns=["user","artist","track","playlist"]
df.head()

,user,artist,track,playlist
0,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello,(The Angels Wanna Wear My) Red Shoes,HARD ROCK 2010
1,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello & The Attractions,"(What's So Funny 'Bout) Peace, Love And Unders...",HARD ROCK 2010
2,9cc0cfd4d7d7885102480dd99e7a90d6,Tiffany Page,7 Years Too Late,HARD ROCK 2010
3,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello & The Attractions,Accidents Will Happen,HARD ROCK 2010
4,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello,Alison,HARD ROCK 2010


In [4]:
# note
df2 = pd.read_csv('../../data/spotify_dataset_2.csv')
df2.head()

,Artist(s),song,text,Length,emotion,Genre,Album,Release Date,Key,Tempo,Loudness (db),Time signature,Explicit,Popularity,Energy,Danceability,Positiveness,Speechiness,Liveness,Acousticness,Instrumentalness,Good for Party,Good for Work/Study,Good for Relaxation/Meditation,Good for Exercise,Good for Running,Good for Yoga/Stretching,Good for Driving,Good for Social Gatherings,Good for Morning Routine,Similar Artist 1,Similar Song 1,Similarity Score 1,Similar Artist 2,Similar Song 2,Similarity Score 2,Similar Artist 3,Similar Song 3,Similarity Score 3
0,!!!,Even When the Waters Cold,Friends told her she was better off at the bot...,03:47,sadness,hip hop,Thr!!!er,29th April 2013,D min,105,-6.85db,4/4,No,40,83,71,87,4,16,11,0,0,0,0,0,0,0,0,0,0,Corey Smith,If I Could Do It Again,0.986061,Toby Keith,Drinks After Work,0.983719,Space,Neighbourhood,0.983236
1,!!!,One Girl / One Boy,"Well I heard it, playing soft From a drunken b...",04:03,sadness,hip hop,Thr!!!er,29th April 2013,A# min,117,-5.75db,4/4,No,42,85,70,87,4,32,0,0,0,0,0,0,0,0,0,0,0,Hiroyuki Sawano,BRE@TH//LESS,0.995409,When In Rome,Heaven Knows,0.990905,Justice Crew,Everybody,0.984483
2,!!!,Pardon My Freedom,"Oh my god, did I just say that out loud? Shoul...",05:51,joy,hip hop,Louden Up Now,8th June 2004,A Maj,121,-6.06db,4/4,No,29,89,71,63,8,64,0,20,0,0,0,1,0,0,0,0,0,Ricky Dillard,More Abundantly Medley Live,0.993176,Juliet,Avalon,0.965147,The Jacksons,Lovely One,0.956752
3,!!!,Ooo,[Verse 1] Remember when I called you on the te...,03:44,joy,hip hop,As If,16th October 2015,A min,122,-5.42db,4/4,No,24,84,78,97,4,12,12,0,0,0,0,1,0,0,0,0,0,Eric Clapton,Man Overboard,0.992749,Roxette,Don't Believe In Accidents,0.991494,Tiwa Savage,My Darlin,0.990381
4,!!!,Freedom 15,[Verse 1] Calling me like I got something to s...,06:00,joy,hip hop,As If,16th October 2015,F min,123,-5.57db,4/4,No,30,71,77,70,7,10,4,1,0,0,0,1,0,0,0,0,0,Cibo Matto,Lint Of Love,0.981610,Barrington Levy,Better Than Gold,0.981524,Freestyle,Its Automatic,0.981415


In [5]:
# see if you can join the two datasets and guage success of join

# clean up artist and track names (create new fields for this)
def clean_text(text):
    """
    Drops spaces and lower-cases text
    """
    return str(text).lower().replace(' ', '')
    # if isinstance(text, str): # Check if the input is a string
    #     return re.sub(r'[^\w\s]', '', text)
    # else:
    #     return text # Return the original value if not a string

# df['artist_track'] = (df['artist'].apply(clean_text) + ' ' + 
#                       df['track'].apply(clean_text))
# df2['artist_track'] = (df2['Artist(s)'].apply(clean_text) + ' ' +
#                        df2['song'].apply(clean_text))
# TODO: instead of merging by track/artist, do artist alone (assumption that general things like genre are same and numbers can be averaged)
df['artist_clean'] = (df['artist'].apply(clean_text))
df2['artist_clean'] = (df2['Artist(s)'].apply(clean_text))

In [6]:
df2.head()

,Artist(s),song,text,Length,emotion,Genre,Album,Release Date,Key,Tempo,Loudness (db),Time signature,Explicit,Popularity,Energy,Danceability,Positiveness,Speechiness,Liveness,Acousticness,Instrumentalness,Good for Party,Good for Work/Study,Good for Relaxation/Meditation,Good for Exercise,Good for Running,Good for Yoga/Stretching,Good for Driving,Good for Social Gatherings,Good for Morning Routine,Similar Artist 1,Similar Song 1,Similarity Score 1,Similar Artist 2,Similar Song 2,Similarity Score 2,Similar Artist 3,Similar Song 3,Similarity Score 3,artist_clean
0,!!!,Even When the Waters Cold,Friends told her she was better off at the bot...,03:47,sadness,hip hop,Thr!!!er,29th April 2013,D min,105,-6.85db,4/4,No,40,83,71,87,4,16,11,0,0,0,0,0,0,0,0,0,0,Corey Smith,If I Could Do It Again,0.986061,Toby Keith,Drinks After Work,0.983719,Space,Neighbourhood,0.983236,!!!
1,!!!,One Girl / One Boy,"Well I heard it, playing soft From a drunken b...",04:03,sadness,hip hop,Thr!!!er,29th April 2013,A# min,117,-5.75db,4/4,No,42,85,70,87,4,32,0,0,0,0,0,0,0,0,0,0,0,Hiroyuki Sawano,BRE@TH//LESS,0.995409,When In Rome,Heaven Knows,0.990905,Justice Crew,Everybody,0.984483,!!!
2,!!!,Pardon My Freedom,"Oh my god, did I just say that out loud? Shoul...",05:51,joy,hip hop,Louden Up Now,8th June 2004,A Maj,121,-6.06db,4/4,No,29,89,71,63,8,64,0,20,0,0,0,1,0,0,0,0,0,Ricky Dillard,More Abundantly Medley Live,0.993176,Juliet,Avalon,0.965147,The Jacksons,Lovely One,0.956752,!!!
3,!!!,Ooo,[Verse 1] Remember when I called you on the te...,03:44,joy,hip hop,As If,16th October 2015,A min,122,-5.42db,4/4,No,24,84,78,97,4,12,12,0,0,0,0,1,0,0,0,0,0,Eric Clapton,Man Overboard,0.992749,Roxette,Don't Believe In Accidents,0.991494,Tiwa Savage,My Darlin,0.990381,!!!
4,!!!,Freedom 15,[Verse 1] Calling me like I got something to s...,06:00,joy,hip hop,As If,16th October 2015,F min,123,-5.57db,4/4,No,30,71,77,70,7,10,4,1,0,0,0,1,0,0,0,0,0,Cibo Matto,Lint Of Love,0.981610,Barrington Levy,Better Than Gold,0.981524,Freestyle,Its Automatic,0.981415,!!!


# Data preprocessing

## clustering artists

Before merging:
- convert Length to seconds
- Genre has multiple entries - figure out if they should be a list, etc
- convert Release Date to date
- need to group by artist and average numerical fields
  - Length
  - Tempo
  - Loudness (db)
  - Popularity
  - Energy
  - Danceability
  - Positiveness
  - Speechiness
  - Liveness
  - Acousticness
  - Instrumentalness
  - Good for Party
  - Good for Work/Study
  - Good for (remaining columns)
- need to group by artist keep top freq of categorical columns
  - Genre
  - Key
  - Time signature
  - Explicit

In [7]:
# convert length into seconds
def convert_to_seconds(time_str):
    minutes, seconds = map(int, time_str.split(':'))
    return minutes * 60 + seconds

df2['Length_Seconds'] = df2['Length'].apply(convert_to_seconds)
# check
# df2[['Length', 'Length_Seconds']].head()

# convert Release Date to date
def convert_to_date(date_str):
    # Remove the ordinal suffix if present (e.g., 'st', 'nd', 'rd', 'th')
    date_str = re.sub(r'(?<=\d)(st|nd|rd|th)', '', date_str)
    
    return pd.to_datetime(date_str, format='%d %B %Y')

df2['Release_Date_Format'] = df2['Release Date'].apply(convert_to_date)
# df2[['Release Date', 'Release_Date_Format']].head()

# drop db from Loudness (db) and make sure it is numerical
df2['loudness_db_num'] = df2['Loudness (db)'].str.replace('db', '').astype('float')
# df2[['Loudness (db)', 'loudness_db_num']]

# drop song similarity (this is going to be artist level)
df2 = df2.drop(['Similarity Score 1', 'Similarity Score 2', 'Similarity Score 3'], 
               axis=1)

In [8]:
# pull all of the numerical columns into a list for averaging
numeric_cols = df2.select_dtypes(include=['number']).columns.tolist()
numeric_cols

# take mean of all numerical columns
df_artist_num = df2.groupby('artist_clean')[numeric_cols].mean().reset_index()
df_artist_num.head()

,artist_clean,Tempo,Popularity,Energy,Danceability,Positiveness,Speechiness,Liveness,Acousticness,Instrumentalness,Good for Party,Good for Work/Study,Good for Relaxation/Meditation,Good for Exercise,Good for Running,Good for Yoga/Stretching,Good for Driving,Good for Social Gatherings,Good for Morning Routine,Length_Seconds,loudness_db_num
0,!!!,118.000000,27.0625,83.312500,73.312500,71.125000,6.437500,18.187500,5.500000,11.6875,0.0,0.0,0.0,0.5,0.0,0.0,0.0,0.0,0.0625,280.500000,-6.65875
1,"!!!,lealea",126.500000,33.5000,82.500000,82.000000,80.000000,6.000000,9.500000,2.500000,16.0000,0.0,0.0,0.0,0.5,0.0,0.0,0.0,0.0,0.0000,275.500000,-6.39000
2,!marc¡,175.000000,0.0000,40.000000,72.000000,52.000000,28.000000,12.000000,0.000000,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0000,122.000000,-7.99000
3,"!yadnus,daylyt",82.333333,4.0000,57.333333,56.666667,88.666667,24.333333,29.666667,51.333333,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0000,81.666667,-10.52000
4,!zeesh,101.000000,51.0000,31.000000,64.000000,81.000000,38.000000,79.000000,85.000000,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0000,112.000000,-13.78000


In [9]:
len(df_artist_num)

127173

In [10]:
# TODO: group by categorical columns and keep top freq
categoric_cols = df2.select_dtypes(include=['object', 'datetime']).columns.tolist()
remove_list = ['Artist(s)', 'Release Date', 'song', 'text', 'Length', 
               'Album', 'Loudness (db)', 'Similar Song 1', 
               'Similar Song 2', 'Similar Song 3']

# Remove multiple specific items by their names using list comprehension
categoric_cols = [s for s in categoric_cols if s not in remove_list]
categoric_cols

# for debugging purposes (remove once you find bad column)
categoric_cols = ['Genre', 'Key']

df_artist_cat = df2.groupby('artist_clean')[categoric_cols].apply(lambda x: x.value_counts().index[0])
# df_artist_cat = df_artist_cat.explode(categoric_cols)

In [11]:
# df_artist_cat.explode(categoric_cols)
# index_series = df_artist_cat.index.to_frame(name='index')
tuple_series = pd.DataFrame.from_records(df_artist_cat.tolist(), columns=categoric_cols)
# df_artist_cat = pd.concat([index_series, tuple_series], axis=1)
# df_artist_cat
len(tuple_series)

127173

In [ ]:
# combine tuple_series and df_artist_num by index
df_artist_info = pd.merge(tuple_series, df_artist_num, left_index=True, right_index=True)
df_artist_info

,Genre,Key,artist_clean,Tempo,Popularity,Energy,Danceability,Positiveness,Speechiness,Liveness,Acousticness,Instrumentalness,Good for Party,Good for Work/Study,Good for Relaxation/Meditation,Good for Exercise,Good for Running,Good for Yoga/Stretching,Good for Driving,Good for Social Gatherings,Good for Morning Routine,Length_Seconds,loudness_db_num
0,hip hop,A Maj,!!!,118.000000,27.0625,83.312500,73.312500,71.125000,6.437500,18.187500,5.500000,11.6875,0.0,0.0,0.0,0.5,0.0,0.0,0.0,0.0,0.0625,280.500000,-6.65875
1,hip hop,C Maj,"!!!,lealea",126.500000,33.5000,82.500000,82.000000,80.000000,6.000000,9.500000,2.500000,16.0000,0.0,0.0,0.0,0.5,0.0,0.0,0.0,0.0,0.0000,275.500000,-6.39000
2,hip hop,C Maj,!marc¡,175.000000,0.0000,40.000000,72.000000,52.000000,28.000000,12.000000,0.000000,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0000,122.000000,-7.99000
3,hip hop,A Maj,"!yadnus,daylyt",82.333333,4.0000,57.333333,56.666667,88.666667,24.333333,29.666667,51.333333,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0000,81.666667,-10.52000
4,hip hop,D Maj,!zeesh,101.000000,51.0000,31.000000,64.000000,81.000000,38.000000,79.000000,85.000000,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0000,112.000000,-13.78000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
127168,house,B min,박혜진parkhyejin,132.000000,37.5000,67.000000,86.000000,21.500000,13.500000,8.500000,21.500000,48.0000,0.0,0.0,0.0,0.5,0.0,0.0,0.0,0.0,0.5000,279.500000,-10.23500
127169,metalcore,G# Maj,심형진hyungjinsim,128.000000,29.0000,43.000000,39.000000,26.000000,3.000000,10.000000,54.000000,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0000,527.000000,-9.68000
127170,hip hop,G# min,우정하,75.000000,21.0000,79.000000,56.000000,41.000000,5.000000,19.000000,0.000000,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0000,233.000000,-4.67000
127171,hip hop,E Maj,제노,126.000000,5.0000,90.000000,59.000000,69.000000,4.000000,7.000000,27.000000,0.0000,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0000,192.000000,-3.93000


In [13]:
# merge is creating more rows, drop duplicate artist_track/artist_clean from df2
# df2 = df2.drop_duplicates(subset=['artist_track'])
# df2 = df2.drop_duplicates(subset=['artist_clean'])

In [14]:
# join datasets and check for missing
print(len(df))
# df = df.merge(df2, how='left', on='artist_track')
# df = df.merge(df2, how='left', on='artist_clean')
df = df.merge(df_artist_info, how='left', on='artist_clean')
print(len(df))

12891680
12891680


In [15]:
df.isnull().sum()

user                                    0
artist                              33568
track                                  85
playlist                             1246
artist_clean                            0
Genre                             3580381
Key                               3580381
Tempo                             3580381
Popularity                        3580381
Energy                            3580381
Danceability                      3580381
Positiveness                      3580381
Speechiness                       3580381
Liveness                          3580381
Acousticness                      3580381
Instrumentalness                  3580381
Good for Party                    3580381
Good for Work/Study               3580381
Good for Relaxation/Meditation    3580381
Good for Exercise                 3580381
Good for Running                  3580381
Good for Yoga/Stretching          3580381
Good for Driving                  3580381
Good for Social Gatherings        

In [16]:
# artist_track
# 8890167 / 12891680

# artist (spaces removed)
3580381 / 12891680

# artist (characters removed)
# 3824768 / 12891680

0.27772803854889355

In [17]:
# checking for missing data (original merge)
df_null = df[df['artist_clean'].isnull()]['artist'].drop_duplicates()
print(len(df_null))
df_null.head()


0


Series([], Name: artist, dtype: object)

In [18]:
df.head()

,user,artist,track,playlist,artist_clean,Genre,Key,Tempo,Popularity,Energy,Danceability,Positiveness,Speechiness,Liveness,Acousticness,Instrumentalness,Good for Party,Good for Work/Study,Good for Relaxation/Meditation,Good for Exercise,Good for Running,Good for Yoga/Stretching,Good for Driving,Good for Social Gatherings,Good for Morning Routine,Length_Seconds,loudness_db_num
0,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello,(The Angels Wanna Wear My) Red Shoes,HARD ROCK 2010,elviscostello,"folk,country,new wave",C# Maj,124.223214,35.071429,41.473214,50.196429,49.37500,4.758929,14.785714,34.071429,0.232143,0.008929,0.089286,0.035714,0.133929,0.026786,0.026786,0.035714,0.0,0.071429,207.482143,-11.26500
1,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello & The Attractions,"(What's So Funny 'Bout) Peace, Love And Unders...",HARD ROCK 2010,elviscostello&theattractions,"folk,country,new wave",D Maj,132.271028,23.934579,63.233645,53.719626,67.46729,4.514019,17.971963,18.467290,1.598131,0.046729,0.028037,0.009346,0.252336,0.112150,0.000000,0.084112,0.0,0.056075,194.093458,-9.11215
2,9cc0cfd4d7d7885102480dd99e7a90d6,Tiffany Page,7 Years Too Late,HARD ROCK 2010,tiffanypage,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello & The Attractions,Accidents Will Happen,HARD ROCK 2010,elviscostello&theattractions,"folk,country,new wave",D Maj,132.271028,23.934579,63.233645,53.719626,67.46729,4.514019,17.971963,18.467290,1.598131,0.046729,0.028037,0.009346,0.252336,0.112150,0.000000,0.084112,0.0,0.056075,194.093458,-9.11215
4,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello,Alison,HARD ROCK 2010,elviscostello,"folk,country,new wave",C# Maj,124.223214,35.071429,41.473214,50.196429,49.37500,4.758929,14.785714,34.071429,0.232143,0.008929,0.089286,0.035714,0.133929,0.026786,0.026786,0.035714,0.0,0.071429,207.482143,-11.26500


In [19]:
df.describe()

,Tempo,Popularity,Energy,Danceability,Positiveness,Speechiness,Liveness,Acousticness,Instrumentalness,Good for Party,Good for Work/Study,Good for Relaxation/Meditation,Good for Exercise,Good for Running,Good for Yoga/Stretching,Good for Driving,Good for Social Gatherings,Good for Morning Routine,Length_Seconds,loudness_db_num
count,9.311299e+06,9.311299e+06,9.311299e+06,9.311299e+06,9.311299e+06,9.311299e+06,9.311299e+06,9.311299e+06,9.311299e+06,9.311299e+06,9.311299e+06,9.311299e+06,9.311299e+06,9.311299e+06,9.311299e+06,9.311299e+06,9.311299e+06,9.311299e+06,9.311299e+06,9.311299e+06
mean,1.213090e+02,3.566842e+01,6.425844e+01,5.427470e+01,4.751223e+01,7.668495e+00,1.990600e+01,2.561995e+01,1.122791e+01,7.054176e-02,8.769304e-02,3.441249e-02,1.785570e-01,4.337825e-02,2.458800e-02,4.205916e-02,1.311602e-02,6.244367e-02,2.423945e+02,-8.199055e+00
std,1.277773e+01,1.264528e+01,1.759296e+01,1.180060e+01,1.585175e+01,6.635699e+00,7.677257e+00,2.276376e+01,1.776854e+01,1.299870e-01,1.562515e-01,8.892282e-02,2.071977e-01,9.022066e-02,7.704091e-02,8.736261e-02,4.789433e-02,1.132643e-01,6.109997e+01,3.114644e+00
min,3.700000e+01,0.000000e+00,0.000000e+00,6.000000e+00,0.000000e+00,2.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.500000e+01,-4.029000e+01
25%,1.146760e+02,2.744444e+01,5.394118e+01,4.650633e+01,3.733796e+01,4.210191e+00,1.552096e+01,8.133603e+00,4.019139e-01,0.000000e+00,0.000000e+00,0.000000e+00,2.521008e-02,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,2.120000e+02,-9.777500e+00
50%,1.213171e+02,3.483333e+01,6.631599e+01,5.350000e+01,4.760606e+01,5.460526e+00,1.890164e+01,1.900000e+01,3.511905e+00,1.758242e-02,2.222222e-02,0.000000e+00,1.185185e-01,1.069519e-02,0.000000e+00,1.250000e-02,0.000000e+00,2.834008e-02,2.355111e+02,-7.616528e+00
75%,1.276667e+02,4.350000e+01,7.660714e+01,6.225000e+01,5.780000e+01,8.000000e+00,2.286813e+01,3.700000e+01,1.460360e+01,8.653846e-02,1.052632e-01,2.857143e-02,2.500000e-01,5.188679e-02,1.515152e-02,5.303030e-02,0.000000e+00,8.000000e-02,2.624444e+02,-5.968681e+00
max,2.000000e+02,8.300000e+01,1.000000e+02,9.800000e+01,9.900000e+01,9.700000e+01,9.900000e+01,1.000000e+02,9.900000e+01,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,2.968000e+03,2.030000e+00


# Data exploration

## Questions

- How many artists do we have?
  - 289,821
- How many users do we have?
  - 15,918
- How many playlists do users have?
  - 15, on average
- What are the top 5 most popular artists?
  - Coldplay
  - Daft Punk
  - Rihanna
  - David Guetta
  - Calvin Harris
- What are the top 5 most popular songs?
  - m83 midnightcity
  - daftpunk getlucky-radioedit
  - imaginedragons radioactive
  - ofmonstersandmen littletalks
  - avicii wakemeup

In [20]:
# number of artists
num_artists = df['artist'].nunique()
num_artists

289821

In [21]:
# number of users
num_users = df['user'].nunique()
num_users

15918

In [22]:
# avg number of playlists per user
playlist_user = df[['user', 'playlist']].drop_duplicates()
playlist_user.head()

,user,playlist
0,9cc0cfd4d7d7885102480dd99e7a90d6,HARD ROCK 2010
67,9cc0cfd4d7d7885102480dd99e7a90d6,IOW 2012
104,07f0fc3be95dcd878966b1f9572ff670,2080
114,07f0fc3be95dcd878966b1f9572ff670,C418
148,07f0fc3be95dcd878966b1f9572ff670,Chill out


In [23]:
# user playlist counts
playlist_user_counts = playlist_user.groupby('user')['playlist'].count()
playlist_user_counts.mean()

14.562382208820203

# Data preprocessing

In [24]:
# create combined artist/track feature
def clean_text(text):
    """
    Drops spaces and lower-cases text
    """
    return str(text).lower().replace(' ', '')

df['artist_track'] = (df['artist'].apply(clean_text) + ' ' + 
                      df['track'].apply(clean_text))
df.head()

,user,artist,track,playlist,artist_clean,Genre,Key,Tempo,Popularity,Energy,Danceability,Positiveness,Speechiness,Liveness,Acousticness,Instrumentalness,Good for Party,Good for Work/Study,Good for Relaxation/Meditation,Good for Exercise,Good for Running,Good for Yoga/Stretching,Good for Driving,Good for Social Gatherings,Good for Morning Routine,Length_Seconds,loudness_db_num,artist_track
0,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello,(The Angels Wanna Wear My) Red Shoes,HARD ROCK 2010,elviscostello,"folk,country,new wave",C# Maj,124.223214,35.071429,41.473214,50.196429,49.37500,4.758929,14.785714,34.071429,0.232143,0.008929,0.089286,0.035714,0.133929,0.026786,0.026786,0.035714,0.0,0.071429,207.482143,-11.26500,elviscostello (theangelswannawearmy)redshoes
1,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello & The Attractions,"(What's So Funny 'Bout) Peace, Love And Unders...",HARD ROCK 2010,elviscostello&theattractions,"folk,country,new wave",D Maj,132.271028,23.934579,63.233645,53.719626,67.46729,4.514019,17.971963,18.467290,1.598131,0.046729,0.028037,0.009346,0.252336,0.112150,0.000000,0.084112,0.0,0.056075,194.093458,-9.11215,elviscostello&theattractions (what'ssofunny'bo...
2,9cc0cfd4d7d7885102480dd99e7a90d6,Tiffany Page,7 Years Too Late,HARD ROCK 2010,tiffanypage,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,tiffanypage 7yearstoolate
3,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello & The Attractions,Accidents Will Happen,HARD ROCK 2010,elviscostello&theattractions,"folk,country,new wave",D Maj,132.271028,23.934579,63.233645,53.719626,67.46729,4.514019,17.971963,18.467290,1.598131,0.046729,0.028037,0.009346,0.252336,0.112150,0.000000,0.084112,0.0,0.056075,194.093458,-9.11215,elviscostello&theattractions accidentswillhappen
4,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello,Alison,HARD ROCK 2010,elviscostello,"folk,country,new wave",C# Maj,124.223214,35.071429,41.473214,50.196429,49.37500,4.758929,14.785714,34.071429,0.232143,0.008929,0.089286,0.035714,0.133929,0.026786,0.026786,0.035714,0.0,0.071429,207.482143,-11.26500,elviscostello alison


In [25]:
# run user counts for each artist_track and each artist - can be used to measure song popularity
track_user_count = df.groupby('artist_track')['user'].count()
track_user_count = pd.DataFrame(track_user_count).reset_index()
track_user_count.columns = ['artist_track', 'count']
track_user_count.sort_values('count', ascending=False).head()

,artist_track,count
1462170,m83 midnightcity,2609
517563,daftpunk getlucky-radioedit,2341
1049832,imaginedragons radioactive,2336
1764670,ofmonstersandmen littletalks,2263
181773,avicii wakemeup,2242


In [26]:
# find most popular artists (count of users with artist in playlist)
artist_user_count = df[['user', 'artist']].drop_duplicates() \
    .groupby('artist')['user'].count()
artist_user_count = pd.DataFrame(artist_user_count).reset_index()
artist_user_count.columns = ['artist', 'count']
artist_user_count.sort_values('count', ascending=False).head()

,artist,count
50551,Coldplay,4645
59037,Daft Punk,4631
210322,Rihanna,4092
63007,David Guetta,3861
39657,Calvin Harris,3732


In [27]:
# create artist_popularity and track_popularity features
# artist popularity
artist_user_count['artist_popularity'] = artist_user_count['count'] / num_users
artist_user_count = artist_user_count[['artist', 'artist_popularity']]
artist_user_count.sort_values('artist_popularity', ascending=False).head()

,artist,artist_popularity
50551,Coldplay,0.291808
59037,Daft Punk,0.290929
210322,Rihanna,0.257067
63007,David Guetta,0.242556
39657,Calvin Harris,0.234452


In [28]:
# track popularity
track_user_count['track_popularity'] = track_user_count['count'] / num_users
track_user_count = track_user_count[['artist_track', 'track_popularity']]
track_user_count.sort_values('track_popularity', ascending=False).head()

,artist_track,track_popularity
1462170,m83 midnightcity,0.163903
517563,daftpunk getlucky-radioedit,0.147066
1049832,imaginedragons radioactive,0.146752
1764670,ofmonstersandmen littletalks,0.142166
181773,avicii wakemeup,0.140847


In [29]:
# merge both with main dataset
# print(len(df))
# df = df.merge(track_user_count, how='left', on='artist_track')
# df = df.merge(artist_user_count, how='left', on='artist')
# print(len(df))
# df.head()

## Model research links

- https://365datascience.com/tutorials/how-to-build-recommendation-system-in-python/


## Initial modeling steps

### MVP model
- combine artist and track to create unique ID for each song
  - vectorize track/artist names with count vectorizer (lengths are similar)

### User similarity
- Look into collaborative filtering, factoring in user IDs
  
### Exploratory
- extract information from playlist names
  - vectorize playlist names to get same "feeling"

In [30]:
# MVP model (only factors in artist-track info)
df_mvp = df[['artist_track']].copy().drop_duplicates()
# sample a smaller amount of data to avoid kernel crash
df_mvp = df_mvp.sample(n=15000, replace=False, random_state=42)
vectorizer = CountVectorizer()
vectorized = vectorizer.fit_transform(df_mvp['artist_track'])
similarities = cosine_similarity(vectorized)

In [31]:
similarities = pd.DataFrame(similarities, 
                            columns=df_mvp['artist_track'], 
                            index=df_mvp['artist_track']).reset_index()
similarities.head()

artist_track,artist_track,rickbraun cadillacslim,robertwells music,ninorota ilpellegrinaggio,whitneyhouston it'snotrightbutit'sokay-club69clubmix,2unlimited unlimitedmegajam,ericdolphy softlyasinamorningsunrise-live,ghostbeach miracle(gigameshremix),"chetbaker,stangetz 3+1=5",bryanferry let'ssticktogether,lessthanjake anti-christ,redhotchilipeppers underthebridge(1992),djkrush ground,thurisaz circadianrhythm,ilovemakonnen imixmy,calltheshots alexistexas,scatmanjohn scatman'sdance,nicolayandkay asthewheelturns,thetroggs 6654321,enigma turnaround,mattnathanson amazingagain,texasinjuly cloudyminds,candyclaws sunarrow,kellyclarkson sinceubeengone(jasonnevinsmixshow),carloscano/raúlalcover quédesespero,larafabian leroiestunefemme,newridersofthepurplesage schooldays-live,katebush rocketman,plateroytu quedemonios!,anjaanaanjaani tujhebhuladiya,quiquegonzalez ¿estuamorenvano?,francescoguccini cirano,nan lorena-manifeellikeawoman.mp3,dispatch mission,javiergarcia llegochango,ofoneblood mindset,otisredding tonofjoy,deveraux stand,frontlineassembly re-birth,thehumanleague&philipwright don'tyouwantme1982,cowboyjunkies angelsinthewilderness,stephansaid thebell,progresia firefirefire-originalmix,shirleybassey whereisthelove-brunoeurubujazzremix,pianotributeplayers takealittleride,relaxingmusic essentialoilonwater-forquietcontemplationandpeaceofmind,gennarorossi purplehaze,thepolyphonicspree section2(it'sthesun)(kcrw)-live,wealltogether wakeupjoe(bonus),petertosh can'tblametheyouth,pistolanniesofthewest stringsromance-extendededitthecivilwarsmixsepcedit,maikamakovski yourreflection,waysted theharshreality,sarahmcmillan downthebanks,charlemagnepalestine tritoneoctave4,tylerward thatkindoflove,philipwesley distantmemory,gatewayworship bound,pier17 lovethemefromthegodfather(mobbeatsmix),mdungu papstouré,sanderkleinenberg mylexicon-radioedit,razorlight monsterboots-itunesversion,lany youarefire,seujorge pessoalparticular,joekelly elephantisland,thetidalsleep thriveandwither,akirakosemura solace,riseagainst peoplelivehere,p.j.pacifico waiting,brandicarlile lovesongs,ivardensphere crackedearth(feat.i:scintilla),artblakey now'sthetime,makenzo makulelefeat.marcus(originalmix),tryptamin thedaywemetonthestaircase,amandablank&santigoldremixes track08,madcobra tekhim,fredperry ifidieyoung-dubstepremix,andystott leaving,sufjanstevens johnwaynegacy.jr.,akitakase wildcatblues,dieform theshadowbox,natkingcole l-o-v-e-multilingualversion,beverleyknight stronghand,thedoors lightmyfire[liveontheedsullivanshow][mono],themarvelettes you'retheoneformebobby-monoversion,kiernanmcmullan heartbeat,"johannsebastianbach goldbergvariations,bwv988:variatio11.a2clav.",thesorentinos loveisall,ryuichisakamoto chansonpourmichele,klangwelt nordland,marvingaye mainthemefromtroubleman-pt.1,hellbillies poker,guillermoanderson primerleón,katebush homeforchristmas,vincekidd mygang,chuckberry runrudolphrun-singleversion,sparklehorse mountains,thefelicebrothers redmustang,"daniellemillet lakmé(2002digitalremaster):duetto:viens,mallika(lakmé/mallika)",dougstanhope priestmolestation,dendriticarbor murmurationend,duman paranoya,phantomplanet leader,orchestralmanoeuvresinthedark ifyouleave,pharoahemonch behindcloseddoors(vocal),kaneholler undertow,penguincafeorchestra paul'sdance-live,thechemicalbrothersvsprimalscream don'tfightcontrol,smoothjazzall-stars partypeople,claude-michelschönberg confrontation,rigo stockholmström,flagofdemocracy trailpass,wahlströms ljusochvärme,redsnapper inyourbacks,"franksinatra oh,whatabeautifulmornin´","elisafranzetti cantataperlanottedinatale:no.31,chorus:amatomiogesù""""","next girl,lady,woman","jordisavall quintettono.4inremaggiorefandango""percordaechitarra(g.448):graveassai-fandango(boccherini)""",circlej kingdomcome,matthewdear inunbending,geppetto&thewhales juno,themightymightybosstones dosomethingcrazy,asleepatthewheel deepwater-dancemix,theapparitions withwolfclotheson,kylerichards theaquariumambientloop,chrispureka afterall,thewoodbrothers losin

In [32]:
input = "2pac theydon'tgiveafuckaboutus"
recommendations = pd.DataFrame(similarities.nlargest(11, input)['artist_track'])
recommendations = recommendations[recommendations['artist_track']!=input]
print(recommendations)

                                            artist_track
2938                                     2pac somuchpain
5016                                   2pac whatzyaphone
9444                         2pac wondawhytheycallubytch
9979                             2pac shortywannabeathug
11112               2pac/snoopdogg 2ofamerikazmostwanted
6200                               2pac pac'slife(remix)
7817                      paulwall theydon'tknow(ft.mike
7026             earth,wind&fire theydon'tsee-remastered
11394  childishgambino theydon'tlikeme(ft.chancethera...
0                                 rickbraun cadillacslim


## MVP model notes

- only factors in name similarity (same artist, titles with same words)
- while artist matches could be accurate, they are also obvious

# Research questions

- Can we identify distinct groups of artists who are enjoyed by similar listeners?

- Can we predict which songs a user is likely to add to their playlist based on their listening history and similar users’ behavior?
  - Engineered fields: Song popularity (Number of times a song appears across all playlists) and Collaborative similarity index (Overlap in playlist content between users)
- Can we predict whether a song will be added to a playlist , based on the song elements (key, tempo, genre, and loudness) of the songs that already exist in the playlist?

In [33]:
df = df.dropna()
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 9310777 entries, 0 to 12891679
Data columns (total 28 columns):
 #   Column                          Dtype  
---  ------                          -----  
 0   user                            object 
 1   artist                          object 
 2   track                           object 
 3   playlist                        object 
 4   artist_clean                    object 
 5   Genre                           object 
 6   Key                             object 
 7   Tempo                           float64
 8   Popularity                      float64
 9   Energy                          float64
 10  Danceability                    float64
 11  Positiveness                    float64
 12  Speechiness                     float64
 13  Liveness                        float64
 14  Acousticness                    float64
 15  Instrumentalness                float64
 16  Good for Party                  float64
 17  Good for Work/Study       

# Clustering to study artists

Since we are not joining metadata based on individual tracks, the study is focused on artists and users.

- Group by user, artist and count number of tracks each user has per artist
- Replace track column with count of tracks by each artist

In [34]:
df.head()

,user,artist,track,playlist,artist_clean,Genre,Key,Tempo,Popularity,Energy,Danceability,Positiveness,Speechiness,Liveness,Acousticness,Instrumentalness,Good for Party,Good for Work/Study,Good for Relaxation/Meditation,Good for Exercise,Good for Running,Good for Yoga/Stretching,Good for Driving,Good for Social Gatherings,Good for Morning Routine,Length_Seconds,loudness_db_num,artist_track
0,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello,(The Angels Wanna Wear My) Red Shoes,HARD ROCK 2010,elviscostello,"folk,country,new wave",C# Maj,124.223214,35.071429,41.473214,50.196429,49.37500,4.758929,14.785714,34.071429,0.232143,0.008929,0.089286,0.035714,0.133929,0.026786,0.026786,0.035714,0.0,0.071429,207.482143,-11.26500,elviscostello (theangelswannawearmy)redshoes
1,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello & The Attractions,"(What's So Funny 'Bout) Peace, Love And Unders...",HARD ROCK 2010,elviscostello&theattractions,"folk,country,new wave",D Maj,132.271028,23.934579,63.233645,53.719626,67.46729,4.514019,17.971963,18.467290,1.598131,0.046729,0.028037,0.009346,0.252336,0.112150,0.000000,0.084112,0.0,0.056075,194.093458,-9.11215,elviscostello&theattractions (what'ssofunny'bo...
3,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello & The Attractions,Accidents Will Happen,HARD ROCK 2010,elviscostello&theattractions,"folk,country,new wave",D Maj,132.271028,23.934579,63.233645,53.719626,67.46729,4.514019,17.971963,18.467290,1.598131,0.046729,0.028037,0.009346,0.252336,0.112150,0.000000,0.084112,0.0,0.056075,194.093458,-9.11215,elviscostello&theattractions accidentswillhappen
4,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello,Alison,HARD ROCK 2010,elviscostello,"folk,country,new wave",C# Maj,124.223214,35.071429,41.473214,50.196429,49.37500,4.758929,14.785714,34.071429,0.232143,0.008929,0.089286,0.035714,0.133929,0.026786,0.026786,0.035714,0.0,0.071429,207.482143,-11.26500,elviscostello alison
5,9cc0cfd4d7d7885102480dd99e7a90d6,Lissie,All Be Okay,HARD ROCK 2010,lissie,"rock,folk,country",A min,129.925000,25.675000,47.475000,50.000000,29.27500,3.600000,15.925000,35.650000,1.375000,0.000000,0.025000,0.000000,0.100000,0.025000,0.000000,0.050000,0.0,0.100000,237.125000,-7.69900,lissie allbeokay


In [35]:
# group by user, artist, and playlist
df_artist_playlist_counts = df.groupby(['user', 'artist', 'playlist'])['track'].count().reset_index()
df_artist_playlist_counts.head()

,user,artist,playlist,track
0,00055176fea33f6e027cd3302289378b,5 Seconds Of Summer,favs,10
1,00055176fea33f6e027cd3302289378b,Against The Current,favs,3
2,00055176fea33f6e027cd3302289378b,All Time Low,favs,8
3,00055176fea33f6e027cd3302289378b,Austin Mahone,favs,1
4,00055176fea33f6e027cd3302289378b,Avril Lavigne,favs,2


In [36]:
# select everything but track from df and drop duplicates to join with dataset above
df_no_track = df.drop(['track'], axis=1).copy()
df_no_track = df_no_track.drop_duplicates()
df_no_track.head()

,user,artist,playlist,artist_clean,Genre,Key,Tempo,Popularity,Energy,Danceability,Positiveness,Speechiness,Liveness,Acousticness,Instrumentalness,Good for Party,Good for Work/Study,Good for Relaxation/Meditation,Good for Exercise,Good for Running,Good for Yoga/Stretching,Good for Driving,Good for Social Gatherings,Good for Morning Routine,Length_Seconds,loudness_db_num,artist_track
0,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello,HARD ROCK 2010,elviscostello,"folk,country,new wave",C# Maj,124.223214,35.071429,41.473214,50.196429,49.37500,4.758929,14.785714,34.071429,0.232143,0.008929,0.089286,0.035714,0.133929,0.026786,0.026786,0.035714,0.0,0.071429,207.482143,-11.26500,elviscostello (theangelswannawearmy)redshoes
1,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello & The Attractions,HARD ROCK 2010,elviscostello&theattractions,"folk,country,new wave",D Maj,132.271028,23.934579,63.233645,53.719626,67.46729,4.514019,17.971963,18.467290,1.598131,0.046729,0.028037,0.009346,0.252336,0.112150,0.000000,0.084112,0.0,0.056075,194.093458,-9.11215,elviscostello&theattractions (what'ssofunny'bo...
3,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello & The Attractions,HARD ROCK 2010,elviscostello&theattractions,"folk,country,new wave",D Maj,132.271028,23.934579,63.233645,53.719626,67.46729,4.514019,17.971963,18.467290,1.598131,0.046729,0.028037,0.009346,0.252336,0.112150,0.000000,0.084112,0.0,0.056075,194.093458,-9.11215,elviscostello&theattractions accidentswillhappen
4,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello,HARD ROCK 2010,elviscostello,"folk,country,new wave",C# Maj,124.223214,35.071429,41.473214,50.196429,49.37500,4.758929,14.785714,34.071429,0.232143,0.008929,0.089286,0.035714,0.133929,0.026786,0.026786,0.035714,0.0,0.071429,207.482143,-11.26500,elviscostello alison
5,9cc0cfd4d7d7885102480dd99e7a90d6,Lissie,HARD ROCK 2010,lissie,"rock,folk,country",A min,129.925000,25.675000,47.475000,50.000000,29.27500,3.600000,15.925000,35.650000,1.375000,0.000000,0.025000,0.000000,0.100000,0.025000,0.000000,0.050000,0.0,0.100000,237.125000,-7.69900,lissie allbeokay


In [37]:
# join data
df_no_track = df_no_track.merge(df_artist_playlist_counts, 
                                on=['user', 'artist', 'playlist'])
df_no_track.head()

,user,artist,playlist,artist_clean,Genre,Key,Tempo,Popularity,Energy,Danceability,Positiveness,Speechiness,Liveness,Acousticness,Instrumentalness,Good for Party,Good for Work/Study,Good for Relaxation/Meditation,Good for Exercise,Good for Running,Good for Yoga/Stretching,Good for Driving,Good for Social Gatherings,Good for Morning Routine,Length_Seconds,loudness_db_num,artist_track,track
0,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello,HARD ROCK 2010,elviscostello,"folk,country,new wave",C# Maj,124.223214,35.071429,41.473214,50.196429,49.37500,4.758929,14.785714,34.071429,0.232143,0.008929,0.089286,0.035714,0.133929,0.026786,0.026786,0.035714,0.0,0.071429,207.482143,-11.26500,elviscostello (theangelswannawearmy)redshoes,3
1,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello,HARD ROCK 2010,elviscostello,"folk,country,new wave",C# Maj,124.223214,35.071429,41.473214,50.196429,49.37500,4.758929,14.785714,34.071429,0.232143,0.008929,0.089286,0.035714,0.133929,0.026786,0.026786,0.035714,0.0,0.071429,207.482143,-11.26500,elviscostello alison,3
2,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello,HARD ROCK 2010,elviscostello,"folk,country,new wave",C# Maj,124.223214,35.071429,41.473214,50.196429,49.37500,4.758929,14.785714,34.071429,0.232143,0.008929,0.089286,0.035714,0.133929,0.026786,0.026786,0.035714,0.0,0.071429,207.482143,-11.26500,elviscostello trampthedirtdown,3
3,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello & The Attractions,HARD ROCK 2010,elviscostello&theattractions,"folk,country,new wave",D Maj,132.271028,23.934579,63.233645,53.719626,67.46729,4.514019,17.971963,18.467290,1.598131,0.046729,0.028037,0.009346,0.252336,0.112150,0.000000,0.084112,0.0,0.056075,194.093458,-9.11215,elviscostello&theattractions (what'ssofunny'bo...,3
4,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello & The Attractions,HARD ROCK 2010,elviscostello&theattractions,"folk,country,new wave",D Maj,132.271028,23.934579,63.233645,53.719626,67.46729,4.514019,17.971963,18.467290,1.598131,0.046729,0.028037,0.009346,0.252336,0.112150,0.000000,0.084112,0.0,0.056075,194.093458,-9.11215,elviscostello&theattractions accidentswillhappen,3


In [38]:
df_no_track[df_no_track['artist']=='2pac']

,user,artist,playlist,artist_clean,Genre,Key,Tempo,Popularity,Energy,Danceability,Positiveness,Speechiness,Liveness,Acousticness,Instrumentalness,Good for Party,Good for Work/Study,Good for Relaxation/Meditation,Good for Exercise,Good for Running,Good for Yoga/Stretching,Good for Driving,Good for Social Gatherings,Good for Morning Routine,Length_Seconds,loudness_db_num,artist_track,track
34001,db937456654d2465292c4daa947c95de,2pac,Strane,2pac,"hip-hop,hip hop",D Maj,102.176744,50.27907,73.144186,77.12093,63.669767,23.325581,19.093023,13.060465,0.413953,0.074419,0.009302,0.0,0.097674,0.07907,0.0,0.148837,0.0,0.046512,258.804651,-7.310791,2pac 2ofamericazmostwanted-snoopdoggydogg,36
34002,db937456654d2465292c4daa947c95de,2pac,Strane,2pac,"hip-hop,hip hop",D Maj,102.176744,50.27907,73.144186,77.12093,63.669767,23.325581,19.093023,13.060465,0.413953,0.074419,0.009302,0.0,0.097674,0.07907,0.0,0.148837,0.0,0.046512,258.804651,-7.310791,2pac changedman,36
34003,db937456654d2465292c4daa947c95de,2pac,Strane,2pac,"hip-hop,hip hop",D Maj,102.176744,50.27907,73.144186,77.12093,63.669767,23.325581,19.093023,13.060465,0.413953,0.074419,0.009302,0.0,0.097674,0.07907,0.0,0.148837,0.0,0.046512,258.804651,-7.310791,2pac ghettostar,36
34004,db937456654d2465292c4daa947c95de,2pac,Strane,2pac,"hip-hop,hip hop",D Maj,102.176744,50.27907,73.144186,77.12093,63.669767,23.325581,19.093023,13.060465,0.413953,0.074419,0.009302,0.0,0.097674,0.07907,0.0,0.148837,0.0,0.046512,258.804651,-7.310791,2pac godblessthedead,36
34005,db937456654d2465292c4daa947c95de,2pac,Strane,2pac,"hip-hop,hip hop",D Maj,102.176744,50.27907,73.144186,77.12093,63.669767,23.325581,19.093023,13.060465,0.413953,0.074419,0.009302,0.0,0.097674,0.07907,0.0,0.148837,0.0,0.046512,258.804651,-7.310791,"2pac gotmymindmadeup-featuringdaz,kurupt,redma...",36
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5887676,0e473c6265dccef3aae49096c49d9cb8,2pac,2Pac,2pac,"hip-hop,hip hop",D Maj,102.176744,50.27907,73.144186,77.12093,63.669767,23.325581,19.093023,13.060465,0.413953,0.074419,0.009302,0.0,0.097674,0.07907,0.0,0.148837,0.0,0.046512,258.804651,-7.310791,2pac dearmama,3
5887677,0e473c6265dccef3aae49096c49d9cb8,2pac,2Pac,2pac,"hip-hop,hip hop",D Maj,102.176744,50.27907,73.144186,77.12093,63.669767,23.325581,19.093023,13.060465,0.413953,0.074419,0.009302,0.0,0.097674,0.07907,0.0,0.148837,0.0,0.046512,258.804651,-7.310791,2pac thugzmansion(acoustic)ft.nas,3
5992724,39c7b42f0898d64be970001532ae7d54,2pac,CD1,2pac,"hip-hop,hip hop",D Maj,102.176744,50.27907,73.144186,77.12093,63.669767,23.325581,19.093023,13.060465,0.413953,0.074419,0.009302,0.0,0.097674,0.07907,0.0,0.148837,0.0,0.046512,258.804651,-7.310791,2pac thugsmansion,1
7590663,3887723eed4c92d58b9e997097bda889,2pac,Starred,2pac,"hip-hop,hip hop",D Maj,102.176744,50.27907,73.144186,77.12093,63.669767,23.325581,19.093023,13.060465,0.413953,0.074419,0.009302,0.0,0.097674,0.07907,0.0,0.148837,0.0,0.046512,258.804651,-7.310791,2pac dearmama,1
